# Interaktywna Wizualizacja Uprawnień Looker

Poniższy kod wczytuje plik `looker_graph_all.json` wygenerowany przez skrypt w Pythonie i tworzy wysoce interaktywny graf z użyciem silnika `Vis.js`. 

**Funkcje:**
- Kliknij na węzeł (np. na Grupę), aby podświetlić pełną ścieżkę dostępu (od Modelu po Grupę) i wygasić resztę.
- Najedź na węzeł, by zobaczyć szczegóły (w tym listę użytkowników przypisanych do grupy).
- Możesz swobodnie przesuwać węzły i przybliżać graf.

In [ ]:
import json
from IPython.display import HTML, display

# 1. Wczytanie danych
file_name = "looker_graph_all.json" # Zmień na swój wygenerowany plik
try:
    with open(file_name, "r") as f:
        graph_data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Plik {file_name} nie istnieje! Uruchom najpierw: python3 permissions_graph_extractor.py")

# 2. Przygotowanie danych dla vis.js
nodes = []
edges = []

type_colors = {
    "model": "#ff4d4d",
    "explore": "#ffa64d",
    "dashboard": "#4d4dff",
    "group": "#ff4dff",
    "user": "#8B4513",
    "folder": "#4dff4d",
    "model_set": "#8B00FF",
    "role": "#00FFFF",
    "user_attribute": "#FFC0CB",
    "access_grant": "#FFFF00"
}

level_mapping = {
    "model": 0,
    "explore": 1,
    "dashboard": 2,
    "group": 3
}

for node in graph_data.get("nodes", []):
    n_type = node.get("type", "unknown")
    level = level_mapping.get(n_type, 4)
    
    label = node.get("label", node["id"])
    title_html = f"<b>Typ:</b> {n_type}<br><b>Nazwa:</b> {label}"
    
    members = node.get("members", [])
    if members:
        members_list = "<br>   - ".join(members[:30])
        if len(members) > 30:
            members_list += f"<br>   - ... i {len(members)-30} więcej"
        title_html += f"<br><br><b>Członkowie:</b><br>   - {members_list}"
        
    nodes.append({
        "id": node["id"],
        "label": label[:20] + "..." if len(label) > 20 else label,
        "title": title_html,
        "color": {
            "background": type_colors.get(n_type, "#cccccc"),
            "border": "#333333"
        },
        "level": level,
        "group": n_type
    })

for edge in graph_data.get("edges", []):
    edges.append({
        "id": f"edge_{edge['source']}_{edge['target']}",
        "from": edge["source"],
        "to": edge["target"],
        "arrows": "to"
    })

print(f"Przetworzono {len(nodes)} węzłów i {len(edges)} krawędzi.")

In [ ]:
# 3. Wygenerowanie interaktywnego pliku HTML z kodem JavaScript (vis.js)
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
    <style type="text/css">
        body, html {{ margin: 0; padding: 0; }}
        #mynetwork {{
            width: 100%;
            height: 800px;
            border: 1px solid lightgray;
            background-color: #f9f9f9;
        }}
        .vis-tooltip {{
            font-family: sans-serif;
            font-size: 14px;
            padding: 10px;
            border-radius: 5px;
        }}
    </style>
</head>
<body>
<div id="mynetwork"></div>
<script type="text/javascript">
    var nodes = new vis.DataSet({json.dumps(nodes)});
    var edges = new vis.DataSet({json.dumps(edges)});

    var container = document.getElementById('mynetwork');
    var data = {{
        nodes: nodes,
        edges: edges
    }};
    var options = {{
        layout: {{
            hierarchical: {{
                direction: 'UD',
                sortMethod: 'directed',
                nodeSpacing: 150,
                levelSeparation: 150,
                parentCentralization: true
            }}
        }},
        physics: false,
        interaction: {{
            hover: true,
            tooltipDelay: 100
        }},
        nodes: {{
            shape: 'box',
            font: {{
                size: 14,
                face: 'sans-serif',
                color: '#ffffff'
            }},
            shadow: true
        }},
        edges: {{
            color: {{ color: '#888888', highlight: '#333333' }},
            smooth: {{
                type: 'cubicBezier',
                forceDirection: 'vertical',
                roundness: 0.4
            }}
        }}
    }};
    var network = new vis.Network(container, data, options);

    // Logika podświetlania ścieżek po kliknięciu
    network.on("click", function(params) {{
        if (params.nodes.length > 0) {{
            var clickedNode = params.nodes[0];
            
            var highlightNodes = new Set();
            highlightNodes.add(clickedNode);
            
            function traceUp(nodeId) {{
                var connectedEdges = network.getConnectedEdges(nodeId);
                connectedEdges.forEach(function(edgeId) {{
                    var edge = edges.get(edgeId);
                    if (edge.to === nodeId && !highlightNodes.has(edge.from)) {{
                        highlightNodes.add(edge.from);
                        traceUp(edge.from);
                    }}
                }});
            }}
            
            function traceDown(nodeId) {{
                var connectedEdges = network.getConnectedEdges(nodeId);
                connectedEdges.forEach(function(edgeId) {{
                    var edge = edges.get(edgeId);
                    if (edge.from === nodeId && !highlightNodes.has(edge.to)) {{
                        highlightNodes.add(edge.to);
                        traceDown(edge.to);
                    }}
                }});
            }}
            
            traceUp(clickedNode);
            traceDown(clickedNode);
            
            // Aktualizacja wizualna węzłów i krawędzi
            var updateNodes = [];
            nodes.forEach(function(node) {{
                if (highlightNodes.has(node.id)) {{
                    updateNodes.push({{id: node.id, color: {{opacity: 1}}, font: {{color: 'white'}}}});
                }} else {{
                    updateNodes.push({{id: node.id, color: {{opacity: 0.1}}, font: {{color: 'rgba(255,255,255,0.1)'}}}});
                }}
            }});
            nodes.update(updateNodes);
            
            var updateEdges = [];
            edges.forEach(function(edge) {{
                if (highlightNodes.has(edge.from) && highlightNodes.has(edge.to)) {{
                    updateEdges.push({{id: edge.id, color: {{opacity: 1, color: '#333333'}}, width: 3}});
                }} else {{
                    updateEdges.push({{id: edge.id, color: {{opacity: 0.05, color: '#cccccc'}}, width: 1}});
                }}
            }});
            edges.update(updateEdges);
            
        }} else {{
            // Reset w przypadku kliknięcia w tło
            var resetNodes = [];
            nodes.forEach(function(node) {{
                resetNodes.push({{id: node.id, color: {{opacity: 1}}, font: {{color: 'white'}}}});
            }});
            nodes.update(resetNodes);
            
            var resetEdges = [];
            edges.forEach(function(edge) {{
                resetEdges.push({{id: edge.id, color: {{opacity: 1, color: '#888888'}}, width: 1}});
            }});
            edges.update(resetEdges);
        }}
    }});
</script>
</body>
</html>
"""

with open("interactive_graph.html", "w") as f:
    f.write(html_content)

display(HTML("<iframe src='interactive_graph.html' width='100%' height='850px' frameborder='0'></iframe>"))